# Analyze Author Name Mismatches

Quantifies the impact of improved name normalization in `CreateParsedNamesNormalized` on existing author-work assignments.

For each assignment in `work_authors` (where `author_id IS NOT NULL`), compares the work's `raw_author_name` against the author profile's `full_name` using the same 8 name matching patterns from `MatchAuthors`. Flags assignments where no pattern matches — these are candidates for author profile splitting.

**Prerequisite:** `CreateParsedNamesNormalized` must have been run with the improved parser so both work names and author `full_name` values reflect the new normalization.

### Cell 1: Build mismatch detection table

Joins `work_authors` → `authors` → `parsed_names_normalized` (twice) and applies all 8 name matching patterns.

In [0]:
CREATE OR REPLACE TABLE openalex.authors.name_mismatch_analysis AS
WITH assigned_works AS (
    SELECT
        wa.work_id,
        wa.author_sequence,
        wa.author_id,
        wa.raw_author_name,
        a.full_name AS author_full_name
    FROM openalex.works.work_authors wa
    INNER JOIN openalex.authors.authors a ON wa.author_id = a.id
    WHERE wa.author_id IS NOT NULL
      AND wa.raw_author_name IS NOT NULL
      AND TRIM(wa.raw_author_name) != ''
),
with_both_parses AS (
    SELECT
        aw.work_id,
        aw.author_sequence,
        aw.author_id,
        aw.raw_author_name,
        aw.author_full_name,
        -- Work name parsed
        wpn.normalized_first AS w_first,
        wpn.normalized_first_initial AS w_first_initial,
        wpn.normalized_middle AS w_middle,
        wpn.normalized_middle_initials AS w_middle_initials,
        wpn.normalized_last AS w_last,
        -- Author profile name parsed
        apn.normalized_first AS a_first,
        apn.normalized_first_initial AS a_first_initial,
        apn.normalized_middle AS a_middle,
        apn.normalized_middle_initials AS a_middle_initials,
        apn.normalized_last AS a_last
    FROM assigned_works aw
    LEFT JOIN openalex.authors.parsed_names_normalized wpn
        ON TRIM(aw.raw_author_name) = wpn.raw_author_name
    LEFT JOIN openalex.authors.parsed_names_normalized apn
        ON TRIM(aw.author_full_name) = apn.raw_author_name
),
with_matching AS (
    SELECT
        *,
        -- Block keys
        CASE WHEN w_last IS NULL THEN NULL
             WHEN w_first_initial IS NULL THEN w_last
             ELSE CONCAT(w_first_initial, ' ', w_last)
        END AS work_block_key,
        CASE WHEN a_last IS NULL THEN NULL
             WHEN a_first_initial IS NULL THEN a_last
             ELSE CONCAT(a_first_initial, ' ', a_last)
        END AS author_block_key,

        -- Pattern 1: Exact Full Name (both have full first, full middle, same last)
        (w_first IS NOT NULL AND w_middle IS NOT NULL AND a_first IS NOT NULL AND a_middle IS NOT NULL
         AND w_first = a_first AND w_middle = a_middle AND w_last = a_last
        ) AS pattern_1,

        -- Pattern 2: Exact First, Middle Initial match
        (w_first IS NOT NULL AND w_middle IS NULL AND w_middle_initials IS NOT NULL
         AND a_first IS NOT NULL
         AND w_first = a_first AND w_last = a_last
         AND (a_middle_initials IS NULL OR SUBSTRING(w_middle_initials, 1, 1) = SUBSTRING(a_middle_initials, 1, 1))
        ) AS pattern_2,

        -- Pattern 3: Initials match to Full
        (w_first IS NULL AND w_first_initial IS NOT NULL AND w_middle_initials IS NOT NULL
         AND a_first IS NOT NULL AND a_middle_initials IS NOT NULL
         AND w_first_initial = a_first_initial
         AND SUBSTRING(w_middle_initials, 1, 1) = SUBSTRING(a_middle_initials, 1, 1)
         AND w_last = a_last
        ) AS pattern_3,

        -- Pattern 4: Both have only initials
        (w_first IS NULL AND w_first_initial IS NOT NULL AND a_first IS NULL AND a_first_initial IS NOT NULL
         AND w_middle IS NULL AND w_middle_initials IS NOT NULL AND a_middle IS NULL AND a_middle_initials IS NOT NULL
         AND w_first_initial = a_first_initial
         AND SUBSTRING(w_middle_initials, 1, 1) = SUBSTRING(a_middle_initials, 1, 1)
         AND w_last = a_last
        ) AS pattern_4,

        -- Pattern 5: Exact First + Last, no middle
        (w_first IS NOT NULL AND a_first IS NOT NULL
         AND w_first = a_first AND w_last = a_last
         AND w_middle_initials IS NULL
        ) AS pattern_5,

        -- Pattern 6: First Initial to Full
        (w_first IS NULL AND w_first_initial IS NOT NULL AND w_middle_initials IS NULL
         AND a_first IS NOT NULL
         AND w_first_initial = a_first_initial AND w_last = a_last
        ) AS pattern_6,

        -- Pattern 7: Both have only first initial, no middle
        (w_first IS NULL AND w_first_initial IS NOT NULL AND a_first IS NULL AND a_first_initial IS NOT NULL
         AND w_middle_initials IS NULL AND a_middle_initials IS NULL
         AND w_first_initial = a_first_initial AND w_last = a_last
        ) AS pattern_7,

        -- Pattern 8: Full Name to Initial
        (w_first IS NOT NULL AND a_first IS NULL AND a_first_initial IS NOT NULL
         AND w_first_initial = a_first_initial AND w_last = a_last
        ) AS pattern_8
    FROM with_both_parses
)
SELECT
    work_id, author_sequence, author_id, raw_author_name, author_full_name,
    w_first, w_first_initial, w_middle, w_middle_initials, w_last,
    a_first, a_first_initial, a_middle, a_middle_initials, a_last,
    work_block_key, author_block_key,
    (work_block_key IS NOT NULL AND author_block_key IS NOT NULL
     AND work_block_key = author_block_key) AS block_key_matches,
    (pattern_1 OR pattern_2 OR pattern_3 OR pattern_4 OR pattern_5
     OR pattern_6 OR pattern_7 OR pattern_8) AS any_pattern_matches,
    pattern_1, pattern_2, pattern_3, pattern_4, pattern_5, pattern_6, pattern_7, pattern_8
FROM with_matching

### Cell 2: Summary counts

In [0]:
SELECT
    COUNT(*) AS total_assignments,
    COUNT_IF(NOT block_key_matches) AS block_key_mismatches,
    COUNT_IF(block_key_matches AND NOT any_pattern_matches) AS block_match_pattern_mismatch,
    COUNT_IF(NOT any_pattern_matches) AS total_mismatches,
    COUNT_IF(any_pattern_matches) AS still_valid,
    ROUND(COUNT_IF(NOT any_pattern_matches) * 100.0 / COUNT(*), 4) AS mismatch_pct
FROM openalex.authors.name_mismatch_analysis

### Cell 3: Author-level impact

How many distinct authors are affected and the distribution of mismatched works per author.

In [0]:
SELECT
    COUNT(DISTINCT author_id) AS affected_authors,
    SUM(mismatched_works) AS total_affected_works,
    PERCENTILE_APPROX(mismatched_works, ARRAY(0.25, 0.5, 0.75, 0.9, 0.99)) AS works_per_author_percentiles
FROM (
    SELECT author_id, COUNT(*) AS mismatched_works
    FROM openalex.authors.name_mismatch_analysis
    WHERE NOT any_pattern_matches
    GROUP BY author_id
)

### Cell 4: Mismatch type breakdown

Categorize mismatches to understand which parser improvements caused the most impact.

In [0]:
SELECT
    CASE
        WHEN w_last IS NOT NULL AND a_last IS NOT NULL AND w_last != a_last THEN 'last_name_swap'
        WHEN w_first_initial IS NOT NULL AND a_first_initial IS NOT NULL AND w_first_initial != a_first_initial THEN 'first_initial_change'
        WHEN NOT block_key_matches THEN 'block_key_other'
        WHEN block_key_matches AND NOT any_pattern_matches THEN 'pattern_level_only'
        ELSE 'unparseable'
    END AS mismatch_type,
    COUNT(*) AS count,
    COUNT(DISTINCT author_id) AS distinct_authors
FROM openalex.authors.name_mismatch_analysis
WHERE NOT any_pattern_matches
GROUP BY 1
ORDER BY count DESC

### Cell 5: Author size distribution

Bucket affected authors by total works_count to understand the profile sizes being split.

In [0]:
SELECT
    CASE
        WHEN total_works <= 5 THEN '1-5'
        WHEN total_works <= 20 THEN '6-20'
        WHEN total_works <= 100 THEN '21-100'
        WHEN total_works <= 1000 THEN '101-1000'
        ELSE '1000+'
    END AS author_size_bucket,
    COUNT(*) AS author_count,
    SUM(mismatched_works) AS total_mismatched_works
FROM (
    SELECT
        nma.author_id,
        COALESCE(oa.works_count, 0) AS total_works,
        COUNT(*) AS mismatched_works
    FROM openalex.authors.name_mismatch_analysis nma
    LEFT JOIN openalex.authors.openalex_authors oa ON nma.author_id = oa.id
    WHERE NOT nma.any_pattern_matches
    GROUP BY nma.author_id, oa.works_count
)
GROUP BY 1
ORDER BY 1

### Cell 6: Spot-check examples

Sample mismatched rows for manual review. Focus on last_name_swap cases (likely CJK reorders).

In [0]:
SELECT
    raw_author_name, author_full_name,
    w_first, w_first_initial, w_last,
    a_first, a_first_initial, a_last,
    work_block_key, author_block_key,
    CASE
        WHEN w_last IS NOT NULL AND a_last IS NOT NULL AND w_last != a_last THEN 'last_name_swap'
        WHEN w_first_initial IS NOT NULL AND a_first_initial IS NOT NULL AND w_first_initial != a_first_initial THEN 'first_initial_change'
        WHEN NOT block_key_matches THEN 'block_key_other'
        ELSE 'pattern_level_only'
    END AS mismatch_type,
    author_id
FROM openalex.authors.name_mismatch_analysis
WHERE NOT any_pattern_matches
LIMIT 50

### Cell 7: Pattern-level mismatch detail

For cases where block_key matches but no pattern matches, show why each pattern failed.

In [0]:
SELECT
    raw_author_name, author_full_name,
    w_first, w_middle, w_middle_initials,
    a_first, a_middle, a_middle_initials,
    w_last, a_last,
    pattern_1, pattern_2, pattern_3, pattern_4, pattern_5, pattern_6, pattern_7, pattern_8,
    author_id
FROM openalex.authors.name_mismatch_analysis
WHERE block_key_matches AND NOT any_pattern_matches
LIMIT 50